In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

In [9]:
fraud_df["ip_address"] = fraud_df["ip_address"].astype(float)

ip_df["lower_bound_ip_address"] = ip_df["lower_bound_ip_address"].astype(float)
ip_df["upper_bound_ip_address"] = ip_df["upper_bound_ip_address"].astype(float)

fraud_df = fraud_df.sort_values("ip_address")
ip_df = ip_df.sort_values("lower_bound_ip_address")

merged_df = pd.merge_asof(
    fraud_df,
    ip_df,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

merged_df = merged_df[
    merged_df["ip_address"] <= merged_df["upper_bound_ip_address"]
]

In [26]:
merged_df["signup_time"] = pd.to_datetime(merged_df["signup_time"], errors="coerce")
merged_df["purchase_time"] = pd.to_datetime(merged_df["purchase_time"], errors="coerce")

merged_df["time_since_signup"] = (
    merged_df["purchase_time"] - merged_df["signup_time"]
).dt.total_seconds()

merged_df["hour_of_day"] = merged_df["purchase_time"].dt.hour
merged_df["day_of_week"] = merged_df["purchase_time"].dt.dayofweek

merged_df.drop(["signup_time", "purchase_time"], axis=1, inplace=True)

In [11]:
merged_df["user_txn_count"] = (
    merged_df.groupby("user_id")
    ["user_id"]
    .transform("count")
)

In [12]:
merged_df["device_txn_count"] = (
    merged_df.groupby("device_id")
    ["device_id"]
    .transform("count")
)

In [13]:
merged_df = merged_df.sort_values(
    ["user_id", "purchase_time"]
)

In [14]:
categorical_cols = [
    "source",
    "browser",
    "sex",
    "country"
]

merged_df = pd.get_dummies(
    merged_df,
    columns=categorical_cols,
    drop_first=True
)

In [27]:
X = merged_df.drop("class", axis=1)
y = merged_df["class"]

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numerical_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "user_txn_count",
    "device_txn_count"
]

merged_df[numerical_cols] = scaler.fit_transform(
    merged_df[numerical_cols]
)

In [42]:

X = merged_df.drop(
    columns=[
        "class",
        "user_id",
        "device_id",
        "ip_address",
        # add any other datetime/string ID columns here
    ]
)
y = merged_df["class"]
X = X.select_dtypes(include=["int64", "float64"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [43]:
smote = SMOTE()
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


class
0    93502
1    93502
Name: count, dtype: int64


In [45]:
y_train.value_counts()
y_train_smote.value_counts()

class
0    93502
1    93502
Name: count, dtype: int64

In [46]:
print(X.shape)
print(y.shape)
print(y.value_counts(normalize=True))

(129146, 7)
(129146,)
class
0    0.905007
1    0.094993
Name: proportion, dtype: float64


In [47]:
merged_df.to_csv("../data/processed/fraud_processed.csv", index=False)